# Tema 12 — Clasificación de imágenes con CNN y Transfer Learning

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ecamposv/nlp-vision/blob/main/semana-06/Tema-12/Tema_12.ipynb)

Puedes ejecutar este notebook localmente (VS Code / Jupyter) o en **Google Colab** dando clic en el badge de arriba.

En este notebook entrenamos una **red neuronal convolucional (CNN)** desde cero sobre el dataset **CIFAR-10**, y luego comparamos su desempeño con **transfer learning** y **fine-tuning** usando el modelo preentrenado **EfficientNetB0**.

> 💡 Se recomienda ejecutar en Colab con **GPU** activada (Entorno de ejecución → Cambiar tipo de entorno → GPU) para acelerar el entrenamiento.

**CNN básica entrenada desde cero**

### ¿Qué hace este código?

Entrena una **CNN desde cero** para clasificar las 10 categorías de **CIFAR-10**:

1. **Carga del dataset** CIFAR-10 (60 000 imágenes de 32×32 px) con `tf.keras.datasets`.
2. **Normalización**: los píxeles se escalan de 0–255 a un rango 0–1 para ayudar al entrenamiento.
3. **Arquitectura**: tres bloques **`Conv2D` + `MaxPooling2D`** que extraen características, seguidos de capas densas. El **`Dropout(0.3)`** reduce el sobreajuste y la capa final **`softmax`** entrega la probabilidad de cada una de las 10 clases.
4. **Compilación** con el optimizador **Adam** y la pérdida `sparse_categorical_crossentropy` (etiquetas como enteros).
5. **Entrenamiento** durante 15 épocas reservando un 20 % para validación, y **evaluación** final sobre el conjunto de prueba.

> ⏱️ Este entrenamiento puede tardar varios minutos; con GPU es mucho más rápido.

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

# 1. Cargar dataset CIFAR-10
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()

# 2. Normalizar imágenes
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

# 3. Nombres de las clases
class_names = ['avión', 'automóvil', 'pájaro', 'gato', 'venado',
               'perro', 'rana', 'caballo', 'barco', 'camión']

# 4. Definir CNN básica
cnn_model = models.Sequential([
    layers.Conv2D(32, (3,3), activation='relu', input_shape=(32,32,3)),
    layers.MaxPooling2D((2,2)),

    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D((2,2)),

    layers.Conv2D(128, (3,3), activation='relu'),
    layers.Flatten(),

    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(10, activation='softmax')
])

# 5. Compilar modelo
cnn_model.compile(optimizer='adam',
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])

# 6. Entrenar
history_cnn = cnn_model.fit(
    x_train, y_train,
    epochs=15,
    batch_size=64,
    validation_split=0.2
)

# 7. Evaluar
test_loss, test_acc = cnn_model.evaluate(x_test, y_test)
print("Accuracy CNN básica:", test_acc)

Transfer learning con EfficientNetB0

### ¿Qué hace este código?

Aplica **transfer learning** reutilizando un modelo ya entrenado en millones de imágenes (**ImageNet**):

1. **Redimensiona** las imágenes a 224×224 px, el tamaño que espera **EfficientNetB0**.
2. Carga el modelo base con `include_top=False` (sin la capa de clasificación original) y pesos de **ImageNet**.
3. **Congela** el modelo base (`trainable = False`): se usa solo como **extractor de características**, sin reentrenar sus pesos.
4. Añade una cabeza nueva: `GlobalAveragePooling2D` + `Dropout` + una capa `Dense` **softmax** con 10 salidas.
5. **Compila, entrena** (solo la cabeza) y **evalúa**. Suele lograr mejor precisión que la CNN desde cero con menos épocas.

> ⚠️ Redimensionar todas las imágenes a 224×224 consume bastante memoria; reduce el tamaño del dataset si te quedas sin RAM.

In [ ]:
# 1. Redimensionar imágenes a 224x224
x_train_resized = tf.image.resize(x_train, (224, 224))
x_test_resized = tf.image.resize(x_test, (224, 224))

# 2. Modelo base preentrenado
base_model = tf.keras.applications.EfficientNetB0(
    include_top=False,
    weights='imagenet',
    input_shape=(224, 224, 3)
)

# 3. Congelar extractor de características
base_model.trainable = False

# 4. Construir modelo final
tl_model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.3),
    layers.Dense(10, activation='softmax')
])

# 5. Compilar
tl_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
                 loss='sparse_categorical_crossentropy',
                 metrics=['accuracy'])

# 6. Entrenar
history_tl = tl_model.fit(
    x_train_resized, y_train,
    epochs=8,
    batch_size=32,
    validation_split=0.2
)

# 7. Evaluar
test_loss_tl, test_acc_tl = tl_model.evaluate(x_test_resized, y_test)
print("Accuracy Transfer Learning:", test_acc_tl)

**Fine-tuning parcial**

### ¿Qué hace este código?

Mejora el modelo con **fine-tuning**: ajusta las últimas capas del modelo preentrenado a nuestro problema:

1. **Descongela** el modelo base (`trainable = True`) pero vuelve a congelar todas las capas **excepto las últimas 20**, que son las que se reentrenarán.
2. **Recompila** con un **learning rate muy pequeño** (`1e-5`) para no destruir lo que el modelo ya aprendió.
3. **Reentrena** unas pocas épocas y vuelve a **evaluar**.

> 💡 El fine-tuning suele dar el último empujón de precisión, pero usar un learning rate alto aquí puede empeorar el modelo.

In [ ]:
# Descongelar las últimas capas del modelo base
base_model.trainable = True

for layer in base_model.layers[:-20]:
    layer.trainable = False

# Recompilar con learning rate pequeño
tl_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
                 loss='sparse_categorical_crossentropy',
                 metrics=['accuracy'])

# Reentrenar
history_ft = tl_model.fit(
    x_train_resized, y_train,
    epochs=5,
    batch_size=32,
    validation_split=0.2
)

# Evaluación final
test_loss_ft, test_acc_ft = tl_model.evaluate(x_test_resized, y_test)
print("Accuracy Fine-Tuning:", test_acc_ft)

**Curvas de entrenamiento**

### ¿Qué hace este código?

Define una función para **graficar las curvas de entrenamiento** y la usa para comparar modelos:

1. `plot_history` recibe el historial devuelto por `model.fit` y dibuja dos gráficas: **accuracy** y **loss**.
2. En cada gráfica se comparan las curvas de **entrenamiento** vs **validación** a lo largo de las épocas.
3. Se grafican los resultados de la **CNN básica** y del **Transfer Learning**.

> 🔍 Si la curva de entrenamiento sube pero la de validación se estanca o baja, es señal de **sobreajuste**.

In [ ]:
def plot_history(history, title):
    plt.figure(figsize=(10,4))

    plt.subplot(1,2,1)
    plt.plot(history.history['accuracy'], label='Entrenamiento')
    plt.plot(history.history['val_accuracy'], label='Validación')
    plt.title(f'Accuracy - {title}')
    plt.xlabel('Época')
    plt.ylabel('Accuracy')
    plt.legend()

    plt.subplot(1,2,2)
    plt.plot(history.history['loss'], label='Entrenamiento')
    plt.plot(history.history['val_loss'], label='Validación')
    plt.title(f'Loss - {title}')
    plt.xlabel('Época')
    plt.ylabel('Loss')
    plt.legend()

    plt.show()

plot_history(history_cnn, "CNN básica")
plot_history(history_tl, "Transfer Learning")

**Matriz de confusión y reporte de clasificación**

### ¿Qué hace este código?

Evalúa en detalle el modelo final con una **matriz de confusión** y un **reporte de clasificación**:

1. Genera predicciones sobre el conjunto de prueba y convierte las probabilidades en clases con `np.argmax`.
2. **Matriz de confusión** (`confusion_matrix`): muestra, para cada clase real, cuántas imágenes se predijeron en cada categoría. Se visualiza como un mapa de calor con `seaborn`.
3. **`classification_report`**: imprime **precisión, recall y F1-score** por clase, útil para detectar qué categorías confunde más el modelo.

> 🔍 Los valores fuera de la diagonal de la matriz indican los errores (por ejemplo, confundir *gato* con *perro*).

In [ ]:
# Predicciones con el modelo final
y_pred = tl_model.predict(x_test_resized)
y_pred_classes = np.argmax(y_pred, axis=1)

# Matriz de confusión
cm = confusion_matrix(y_test, y_pred_classes)

plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel("Predicción")
plt.ylabel("Etiqueta real")
plt.title("Matriz de confusión")
plt.show()

# Reporte de clasificación
print(classification_report(y_test, y_pred_classes, target_names=class_names))